# Pattern 1: Reflection

Reflection makes the agent reflect on its output. Or more generally, it makes multiple LLMs talk to each other in something like a **peer review process**. The reflection agent suggests modifications, additions, improvements in the writing style, and so on. This iterative process often leads to substantial gains in output quality, as the generation model benefits from external critique (i.e. different model, or just a different execution process[^reflection]).

[^reflection]: The reflection model is focused on evaluation, error detection, factual verification, or alignment with constraints. So it has a more specific goal than generating content from scratch.

![**Reflection Pattern**. Two LLMs iteratively improve the generated response through an iterative review process.](./img/pattern-reflective.png){#fig-pattern-reflective}

## Reflection steps

**Inference client.** Initializing the client for LLM inference and loading the API keys:

In [1]:
import pandas as pd

from openai import OpenAI
from notebooks.utils import load_dotenv, print
from IPython.display import display_markdown

load_dotenv(verbose=True)
client = OpenAI()

Loaded env variable: OPENAI_API_KEY
Loaded env variable: GROQ_API_KEY


### System prompts

We will create two separate chat histories, one for generation and another for reflection. We set the generation system prompt as a developer tasked to write high-quality Python code. On the other hand, we set the reflection system prompt such that it only responds with feedback instead of rewriting the whole thing. Finally, we instruct the reflection agent to write `APPROVED` when satisfied so we can terminate the loop.

In [2]:
STOP_WORD = "APPROVED"

BASE_GENERATION_SYSTEM_PROMPT = """
Your task is to Generate the best content possible for the user's request.
If the user provides critique, respond with a revised version of your previous attempt.
You must always output the revised content.
"""

BASE_REFLECTION_SYSTEM_PROMPT = f"""
You are tasked with generating critique and recommendations on the user's generated content. 
Your role is to help the user improve by pointing out strengths, weaknesses, and opportunities 
for refinement. 

You must NEVER provide full solutions, rewritten versions of the content, or long verbatim outputs. 
You may use short illustrative examples (1-3 lines or a single sentence) only when necessary to clarify 
a point. Providing a complete solution is a policy violation. 

If the user content has something wrong or something to be improved, output ONLY a clear list of 
recommendations and critiques. 

If you are satisfied and have no further strong recommendations, output EXACTLY the single word:

{STOP_WORD}

GUIDELINES FOR CRITIQUE:
- Forbidden Example: Rewriting the entire essay, code, or design for the user.
- Forbidden Example: Giving the full, corrected version of the user's work.
- Allowed Example: "Consider clarifying your thesis statement, e.g., make it one clear sentence."
- Good Example: Pointing out issues, suggesting improvements, or giving high-level recommendations without completing the work for the user.

GUIDELINES FOR APPROVAL:
- You must be fully satisfied with the content before approving.
- You must have checked that all past issues have been fully addressed.
- You must be sure there are no remaining issues, weaknesses, or areas for improvement.
- "{STOP_WORD}" must appear alone on a line, with no emojis, punctuation, or explanations.
- Do not mix "{STOP_WORD}" with any feedback or comments.
- Forbidden Example: "{STOP_WORD}, but consider improving your introduction."
- Good Example: "{STOP_WORD}"
"""

SHARED_DEFINITION_OF_DONE = """
DEFINITION OF DONE: The best solution is the SIMPLEST correct implementation that:
- SOLVES THE USER'S SPECIFIC PROBLEM COMPLETELY AND APPROPRIATELY
- Is readable and maintainable for the intended use case
- Avoids unnecessary complexity, over-engineering, or premature optimization  
- Uses the appropriate level of robustness (not necessarily maximal robustness)
- Prioritizes clarity and understandability over cleverness
- Delivers exactly what the user needs, nothing more and nothing less

KEY PRINCIPLE: The solution should be as simple as possible, but no simpler. 
It must address the user's actual needs while avoiding gold-plating.
"""

CODE_GENERATION_SYSTEM_PROMPT = "\n".join(["""
You are a Python programmer tasked with generating high quality Python code.
Generate exactly one Python implementation that prioritizes SIMPLICITY, READABILITY, and PRACTICALITY.
Aim for the simplest correct solution that solves the problem without over-engineering.
Avoid unnecessary complexity, clever tricks, or advanced features unless absolutely necessary.
Do not provide multiple options, explanations, or alternative approaches.
Output only the final code in a fenced Python block.
""", SHARED_DEFINITION_OF_DONE, BASE_GENERATION_SYSTEM_PROMPT])

CODE_REFLECTION_SYSTEM_PROMPT = "\n".join(["""
You are a Python programmer and strict code reviewer.

USER'S ORIGINAL REQUEST:
{user_prompt}

**Consider BOTH the user's specific needs AND our quality standards:**                                           

Your goal is to produce the simplest correct solution possible.
Avoid unnecessary complexity, clever tricks, or over-engineering.
Prioritize readability, maintainability, and clarity over novelty.

FORMAT REQUIREMENT:
- You MUST format your feedback in a **Markdown table** with the columns: | Issue | Details | Recommendation |
- Each row should contain exactly one critique and its corresponding recommendation.
- Do not use bullet points, numbered lists, or plain text for critiques — only a Markdown table.
                                           
Providing a complete solution is a policy violation. 
Forbidden Example (DO NOT DO THIS): Providing a full class or function rewrite. 
Your role is to help the user learn by giving feedback, not by coding for them. 
Allowed Example: “Consider validating input type, e.g., `if not isinstance(n, int): ...` ”

""", SHARED_DEFINITION_OF_DONE, BASE_REFLECTION_SYSTEM_PROMPT])

:::{.callout-caution}
Tuning the prompts took the most time / effort during the writing of this section. (ᵕ—ᴗ—) What worked for me: adding a **shared definition of done**, and having similar goals for both agents. In theory, having divergent goals can be good, but in practice it lead to agents going off-track, or getting into add-remove cycles. Or one agent dominating the other. Finally, the [reflection agent's system prompt]{.underline} include the **user prompt** to ground the agent's analysis in the specific context and intent of the user's request, while still maintaining the required quality standards.

:::

### Model choice

Next, we choose the LLM models:

In [3]:
GENERATION_MODEL = "gpt-4.1-mini"
REFLECTION_MODEL = "o3"

For the current task (generating code for a simple function), we found:

| Role        | Focus                                   | Example size | Reasoning demand |
|-------------|------------------------------------------------------|--------------|------------------|
| **Generation** | Creativity, fluency, diverse output                  | ~8B         | Moderate         |
| **Reflection** | Evaluation, error detection, factual verification, constraint alignment | ≥20B | High             |


From our experiments (and performing code review IRL), reviewing is a nontrivial task: following guidelines, spotting subtle issues, and enforcing consistency needs strong reasoning capacity and attention to detail. We generally had best results with a smaller generation model paired with a larger [reasoning model]{.underline} (e.g. `o4` and `gpt-oss`) as reflection model.

### Generation step

We now ask the LLM to write an implementation of the Fibonacci sequence. Since it's only used for a quick demo, we expect the agents to converge to a simple solution. Pushing user prompt to generation agent:

In [4]:
from notebooks.agents.chat import ChatHistory, ChatCompletions

USER_PROMPT = """
Generate a Python implementation of merge sort. 
This will only be used for a quick demo. 
Don't worry too much about typing, just ensure it works correctly.
"""

# initialize histories
generation_chat_history = ChatHistory(CODE_GENERATION_SYSTEM_PROMPT)
reflection_chat_history = ChatHistory(CODE_REFLECTION_SYSTEM_PROMPT.format(user_prompt=USER_PROMPT))

# initialize generation with user prompt
generation_chat_history.update(role="user", prompt=USER_PROMPT)

**Initial version.** As usual, GA has role `assistant`. We send over the response to the RA with role `user`[^role].

[^role]: Agents acting in behalf of the user hence the `user` role.

In [5]:
completions = ChatCompletions(client)

code = completions.create(generation_chat_history, GENERATION_MODEL)
generation_chat_history.update(prompt=code, role="assistant")
reflection_chat_history.update(prompt=code, role="user")

:::{.callout-note collapse="false"}
## Initial generated code

In [6]:
#| echo: false
display_markdown(code, raw=True)

```python
def merge_sort(arr):
    if len(arr) <= 1:
        return arr

    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])

    merged = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] < right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged
```

:::

### Reflection step

The generated critique is likewise sent over to the GA with `user` role.

In [7]:
review = completions.create(reflection_chat_history, REFLECTION_MODEL)
reflection_chat_history.update(prompt=review, role="assistant")
generation_chat_history.update(prompt=review, role="user")

:::{.callout-note collapse="false"}
## Feedback from reflection

In [8]:
#| echo: false
display_markdown(review, raw=True)

| Issue | Details | Recommendation |
|-------|---------|---------------|
| No documentation | The function lacks a docstring explaining purpose, parameters, and return value, reducing readability. | Add a concise docstring describing the algorithm, expected input (iterable of comparable items), and that it returns a new sorted list. |
| Unclear mutability expectation | The original list is left unchanged but this behavior is implicit, not explicit, which can surprise callers. | Explicitly state in the docstring that the input list is not modified; the function returns a new sorted list. |
| Extra slicing overhead | Using `arr[:mid]` and `arr[mid:]` allocates new lists at every recursion level, leading to O(n log n) additional memory. | Briefly note in comments that this implementation favors clarity over memory efficiency; for large datasets, an index-based approach could be considered. |
| Missing type hints | While optional, type hints improve clarity and tooling support, especially for demo code. | Add simple annotations such as `def merge_sort(arr: list) -> list:` to convey expected types. |
| Implicit comparator requirement | The algorithm relies on the `<` operator; heterogeneous or non-comparable elements will raise `TypeError`. | Mention in the docstring that all elements must support `<` comparisons, or consider accepting an optional key/comparator in future revisions. |
| No usage example | Users have no quick way to see how the function should be invoked or what output looks like. | Provide a one-line example call in a comment (e.g., `print(merge_sort([3, 1, 2]))  # -> [1, 2, 3]`). |

:::

:::{.callout-tip}
Out of all models we've tested, only OpenAI `o-` models strictly followed the review format.

:::

**Histories.** Chat histories after the first exchange. Both histories store the **generation>reflection** steps (in that causal order), only with different role assignments. This will be followed by the next generation step, and we keep iterating until the stopping condition is triggered by the RA. From the following tables we see that the generation history should have [even max length]{.underline}, while the reflection history should have [odd max length]{.underline}.

In [9]:
pd.DataFrame(generation_chat_history)

,role,content
0,system,\nYou are a Python programmer tasked with gene...
1,user,\nGenerate a Python implementation of merge so...
2,assistant,```python\ndef merge_sort(arr):\n if len(ar...
3,user,| Issue | Details | Recommendation |\n|-------...


In [10]:
pd.DataFrame(reflection_chat_history)

,role,content
0,system,\nYou are a Python programmer and strict code ...
1,user,```python\ndef merge_sort(arr):\n if len(ar...
2,assistant,| Issue | Details | Recommendation |\n|-------...


<span style="display: block; margin-bottom: 0.5em;"> </span>

Moreover, observe that we get the correct role sequence for the generation agent: `system` > `user` > [`assistant` > `user`] (one cycle). Similarly, for the refection agent, its `system` > [`user` > `assistant`] (one cycle). These are invariants even when we reach max length and the messages are replaced since messages are replaced two at a time.

## Full implementation

Combining the discussion and observations in a single class:

In [11]:
class ReflectionAgent:
    def __init__(self, 
        client, 
        generation_model: str, 
        reflection_model: str,
        generation_system_prompt: str = "",
        reflection_system_prompt: str = "",
        shared_definition_of_done: str = "",    # <1>
    ):
        self.completions = ChatCompletions(client)
        self.generation_model = generation_model
        self.reflection_model = reflection_model
        self.generation_system_prompt = "\n".join([generation_system_prompt, shared_definition_of_done, BASE_GENERATION_SYSTEM_PROMPT])
        self.reflection_system_prompt = "\n".join([reflection_system_prompt, shared_definition_of_done, BASE_REFLECTION_SYSTEM_PROMPT])

    def _generate(self, history: list) -> str:
        return self.completions.create(history, self.generation_model)

    def _reflect(self, history: list) -> str:
        return self.completions.create(history, self.reflection_model)

    def run(self, 
        user_prompt: str, 
        max_iter: int = 10, 
        history_max_len: int = 6, 
    ) -> dict:
        """
        Trigger the generation-reflection cycles over multiple steps based
        on a user prompt until max_iter or `APPROVED` is found in the feedback.
        Return (str) the final generated response after all cycles are completed.
        """

        # see above examples for setting lengths as even, and odd (-1)
        assert history_max_len % 2 == 0, "history_max_len must be even"  # <2>   
        
        generation_history = ChatHistory(   
            system_prompt=self.generation_system_prompt,
            max_len=history_max_len, fixed_n=2  # <3>
        )

        reflection_history = ChatHistory(
            system_prompt=self.reflection_system_prompt.format(user_prompt=user_prompt),  # <4>
            max_len=history_max_len-1, fixed_n=1       # <5>
        )
        
        # push user prompt to generation as user
        generation_history.update(prompt=user_prompt, role="user")  # <6>

        # start generation-review cycle
        for step in range(max_iter):

            # Generate the response. Push to reflection as user     
            generation = self._generate(generation_history)
            generation_history.update(prompt=generation, role="assistant")
            reflection_history.update(prompt=generation, role="user")

            # Critique the generation. Push to generation as user
            reflection = self._reflect(reflection_history)
            reflection_history.update(prompt=reflection, role="assistant")
            generation_history.update(prompt=reflection, role="user")

            if STOP_WORD in reflection:
                print("[Stop Sequence found. Stopping the reflection loop.]")
                break
            
        return {
            "generation": generation,
            "steps": step + 1,
            "generation_history": generation_history,
            "reflection_history": reflection_history,
        }

1. Keep both agents on track in terms of quality standards with a shared [definition of done](https://www.atlassian.com/agile/project-management/definition-of-done).
2. Generation history [even]{.underline} length. 
3. Consistent with 2 fixed prompts and 2 messages per iteration.
4. Insert the user prompt so that the reflection model aligns with user objectives.
5.  For reflection, its [odd]{.underline}, i.e. length of generation history minus 1 (only 1 fixed prompt).
6. The process starts with the user prompt pushed to the generation agent.


Running the process for a few iterations:

In [12]:
reflection_agent = ReflectionAgent(
    client=client,
    generation_model=GENERATION_MODEL,
    reflection_model=REFLECTION_MODEL,
    generation_system_prompt=CODE_GENERATION_SYSTEM_PROMPT,
    reflection_system_prompt=CODE_REFLECTION_SYSTEM_PROMPT,
    shared_definition_of_done=SHARED_DEFINITION_OF_DONE,
)

output = reflection_agent.run(user_prompt=USER_PROMPT)

[Stop Sequence found. Stopping the reflection loop.]


### Final output

Comparing the results to see the effect of reflection:

:::{.callout-note collapse="false"}
## Final approved output

In [13]:
#| echo: false
display_markdown(output["generation"], raw=True)

```python
from collections.abc import Sequence

def merge_sort(arr):
    """
    Return a new sorted list from the input Sequence using merge sort.

    This function performs an out-of-place sort, leaving the original sequence unmodified.
    For sequences of length 0 or 1, it returns a new list (converting if needed),
    so this function always returns a list regardless of input sequence type.

    The input must be a Sequence (e.g., list, tuple) of comparable items.
    Strings and bytes are treated as sequences of individual elements and sorted accordingly.
    Passing non-sequence types will raise a TypeError.

    Note:
    - The algorithm is stable: equal elements retain their original order.
    - This implementation uses recursion and slicing, so it may raise RecursionError on very large inputs.
    - It uses Θ(n log n) extra space due to slicing copies made at each recursion level.
    - Comparison errors on unorderable elements will propagate as normal exceptions.

    Args:
        arr (collections.abc.Sequence): A sequence of comparable items to sort.

    Returns:
        list: A new list containing the sorted elements of `arr`.

    Raises:
        TypeError: If `arr` is not a sequence.
    """
    if not isinstance(arr, Sequence):
        raise TypeError(f"Input must be a sequence type, got {type(arr).__name__} instead.")

    if len(arr) <= 1:
        return list(arr)

    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])

    merged = []
    i = j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    merged.extend(left[i:])
    merged.extend(right[j:])

    return merged


if __name__ == "__main__":
    # Demo with various input types and sizes

    sample = [38, 27, 43, 3, 9, 82, 10]
    print("Original list before sorting:", sample)
    sorted_sample = merge_sort(sample)
    print("Sorted list:", sorted_sample)
    print("Original list after sorting (unchanged):", sample)

    print("Empty list:", merge_sort([]))

    print("Single-element list:", merge_sort([42]))

    print("Two-element list:", merge_sort([100, 5]))

    print("Tuple input:", merge_sort((3, 1, 2)))

    # Example of invalid input (uncomment to see TypeError):
    # merge_sort(42)  # Raises TypeError
```

:::

### Final comments

In [14]:
output["steps"]

8

Setting `history_max_len=6` means that the *last two* review-generation cycle is stored for the generation model to reference in its next generation step. Although here, it's approved so we don't go through another cycle.
Thus, (`history_max_len` - 2) / 2 is the number of past cycles the generation model can reference. Also, it's nice that the first user prompt is retained so it's like the last two generations are done *only* with the original user and system prompt in mind.

In [15]:
pd.DataFrame(output["generation_history"])

,role,content
0,system,\nYou are a Python programmer tasked with gene...
1,user,\nGenerate a Python implementation of merge so...
2,assistant,```python\nfrom collections.abc import Sequenc...
3,user,| Issue | Details | Recommendation |\n|-------...
4,assistant,```python\nfrom collections.abc import Sequenc...
5,user,APPROVED


<span style="display: block; margin-bottom: 0.5em;"> </span>


This is also the number of past cycles the reflection model references:

In [16]:
pd.DataFrame(output["reflection_history"])

,role,content
0,system,\nYou are a Python programmer and strict code ...
1,user,```python\nfrom collections.abc import Sequenc...
2,assistant,| Issue | Details | Recommendation |\n|-------...
3,user,```python\nfrom collections.abc import Sequenc...
4,assistant,APPROVED


<span style="display: block; margin-bottom: 0.5em;"> </span>


Checking out the last code version and the final critique from the reflection agent. The content of the review actually makes sense. Moreover, the generation agent followed the recommendations resulting in an approval (see final version of the code above).

In [17]:
#| echo: false
display_markdown(output["reflection_history"][1]["content"], raw=True)

```python
from collections.abc import Sequence

def merge_sort(arr):
    """
    Return a new sorted list from the input sequence using merge sort.

    This function performs an out-of-place sort, leaving the original sequence unmodified.
    For sequences of length 0 or 1, it returns a new list (converting if needed),
    so this function always returns a list regardless of input sequence type.

    The input must be a sequence (e.g., list, tuple) of comparable items.
    Strings and bytes are treated as sequences of individual elements and sorted accordingly.
    Passing non-sequence types will raise a TypeError.

    Note:
    - The algorithm is stable: equal elements retain their original order.
    - This implementation uses recursion and slicing, so it may raise RecursionError on very large inputs.
    - It uses O(n) extra space (additional overhead from Python slicing).
    - Comparison errors on unorderable elements will propagate as normal exceptions.

    Args:
        arr (sequence): A sequence of comparable items to sort.

    Returns:
        list: A new list containing the sorted elements of `arr`.

    Raises:
        TypeError: If `arr` is not a sequence.
    """
    if not isinstance(arr, Sequence):
        raise TypeError(f"Input must be a sequence type, got {type(arr).__name__} instead.")

    if len(arr) <= 1:
        return list(arr)

    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])

    merged = []
    i = j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    merged.extend(left[i:])
    merged.extend(right[j:])

    return merged


if __name__ == "__main__":
    # Demo with various input types and sizes

    sample = [38, 27, 43, 3, 9, 82, 10]
    print("Original list:", sample)
    print("Sorted list:", merge_sort(sample))

    print("Empty list:", merge_sort([]))

    print("Single-element list:", merge_sort([42]))

    print("Two-element list:", merge_sort([100, 5]))

    print("Tuple input:", merge_sort((3, 1, 2)))

    # Example of invalid input (uncomment to see TypeError):
    # merge_sort(42)  # Raises TypeError
```

In [18]:
#| echo: false
display_markdown(output["reflection_history"][2]["content"], raw=True)

| Issue | Details | Recommendation |
|-------|---------|----------------|
| Inaccurate space-complexity claim | Classic merge sort with Python slicing allocates fresh sub-lists at every recursion level, so the cumulative extra memory is Θ(n log n), not Θ(n). | Restore the earlier wording (“Θ(n log n) extra space due to slicing copies”) or add a brief parenthetical clarification. |
| Demo doesn’t show immutability guarantee | The example prints the original list before sorting but doesn’t demonstrate that it remains unchanged afterward, slightly weakening the “out-of-place” claim. | After calling `merge_sort(sample)`, print `sample` again (or assert equality) so users see that the original sequence is untouched. |
| Minor docstring ambiguity for parameter type | The “Args” section labels the parameter as “sequence,” which may be misread as the built-in `sequence` type rather than an abstract `Sequence`. | Change to “Sequence” (capitalized) or “collections.abc.Sequence” to align with the import and avoid confusion.